# Conditional GANs & Pix2Pix

The first big unlock of 2014-2017 was controlling what a GAN makes. Attach a label, or an image, or a sentence. Pix2Pix did the image version and it still beats every generic text-to-image model on narrow image-to-image tasks.

## Problem definition

An unconditional GAN samples arbitrary faces. Useful for a demo, useless in production. You want: map a sketch to a photo, map a map to an aerial phote, map a daytime scene to nighttime, colorize a grayscale image. In all of these, you are given an input image x and must output y with some semantic correspondence. There are many plausible y per x. Mean-squared error flattens them into mush. An adversarial loss doesn't, because "looks real" is sharp.

Conditinonal GAN add a condition c as an input to both G and D.

Pix2Pix specialized this:

Condition is a full input image, generator is a U-Net, discriminator is a patch-based classifier, and loss is adversarial + L1.

## Basic Concept

Pix2Pix: U-Net generator, PatchGAN discriminator

### Conditional G

`G(x, z) --> y`. In Pix2Pix, `z` is dropout inside `G`, because **the input x is not a noise sampled from standard distribution**.

### Conditional D

`D(x, y) --> [0, 1]`  Input is the pair, This is the key difference: `D` must judge whether `y` is consistent with `x`, not just whether `y` looks real.

### U-Net generator

Encoder-decoder with skip connections across the bottleneck. Cirtical for tasks where input and output share low-level structure (edges, silhouette). Without the skips, high-frequency detail vanishes.

### PatchGAN discriminator

Instead of outputting a single real/fake source, `D` outputs an `NxN` grid where each cell judges a receptive field of ~70x70 pixels. Averaged. 

### Loss

```
loss_G = -log D(x, G(x)) + lambda * ||y - G(x)||_1
loss_D = -log D(x, y) - log (1 - D(x, G(x)))
```

The L1 term stabilize training and pushes G toward the known target. L1 gives sharper edges that L2 (medians, not means).

### CycleGAN -- when you don't have pairs

Pix2Pix needs paired `(x, y)` data. CycleGAN drops this requirement at the cost of an extra loss: the cycle consistency loss. Two generators `G: X-->Y` and `F: Y-->X` （Pick an image from `X or Y`, and generate the oppsite and compair）. Train them so `F(G(x)) = x` and `G(F(y)) = y`. This lets you translate horses to zebras, summer to winter without paired examples.

# Build your Own

## 当前实现：Class 约束怎么引入

下面这个 toy example 用 **2 个离散类别** 控制 1D 高斯的位置：class 0 → 均值 -2，class 1 → 均值 +2。约束通过三条线同时生效。

### 1. 数据：带标签采样

```python
c ~ Uniform{0, 1}
x ~ N(MEANS[c], σ²)      # MEANS = [-2, +2]
```

每个样本都是一对 `(x, c)`。真实分布本身就是条件分布 `p(x | c)`。

### 2. 编码：`one_hot(c)` 拼进输入

类别是离散整数，先转成 one-hot 向量，再与连续量拼接：

| 网络 | 输入拼接 | 维度 |
|------|----------|------|
| **G** | `[z, one_hot(c)]` | `Z_DIM + NUM_CLASSES` = 1 + 2 = 3 |
| **D** | `[x, one_hot(c)]` | `REPRESENT_DIM + NUM_CLASSES` = 1 + 2 = 3 |

`one_hot` 告诉网络「当前是哪个 class」，G 据此决定往哪个 mode 生成，D 据此判断「这个 x 是否像该 class 的真样本」。

### 3. Generator：`G(z, c) → x̂`

- `z ~ N(0, I)` 提供类内随机性（同一 class 每次生成略有不同）
- `c` 决定生成哪个峰（-2 还是 +2）
- 推理时指定 `c` 即可控制输出：`G(z, c=0)` → 靠近 -2，`G(z, c=1)` → 靠近 +2

### 4. Discriminator：`D(x, c) → logit`

D 看的不是单独的 `x`，而是 **(x, c) 配对**：

- 真样本：`D(x_real, c)` → 1（这个 x 在 class c 下看起来真实）
- 假样本：`D(G(z, c), c)` → 0（这个 x 在 class c 下不像真的）

若 G 用 `c=0` 生成了靠近 +2 的样本，D 会判假——因为 `(+2, class=0)` 不匹配。

### 5. 训练循环：同一个 `c` 贯穿 G 和 D

每个 batch：

1. 采样 `(x_real, c)` 
2. D 学：真的 `(x_real, c)` 为 1，假的 `(G(z, c), c)` 为 0
3. G 学：让 `D(G(z, c), c)` 尽量为 1

**关键**：fake 和 real 共用同一组 `cs_t`。G 被迫「按指定 class 造假」，而不是随便造一个 D 能认的样本。

### 和无条件 GAN 的区别

| | 无条件 GAN | 当前 cGAN |
|--|-----------|-----------|
| G 输入 | `z` | `[z, one_hot(c)]` |
| D 输入 | `x` | `[x, one_hot(c)]` |
| 控制生成 | 无法控制 | 指定 `c` 即可 |

In [36]:
import torch
import torch.nn as nn
import random

torch.manual_seed(0)
random.seed(0)

NUM_CLASSES = 2
Z_DIM = 1  # 1D 输出 + 条件 c 已足够；高维 z 易 mode collapse
HIDDEN = 64

REPRESENT_DIM = 1


class Generator(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, z, c, num_classes):
        x = torch.cat([z, torch.nn.functional.one_hot(c, num_classes)], dim=-1)
        return self.net(x)


class Discriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x, c, num_classes):
        x = torch.cat([x, torch.nn.functional.one_hot(c, num_classes)], dim=-1)
        return self.net(x)


G = Generator(NUM_CLASSES + Z_DIM, HIDDEN, REPRESENT_DIM)
D = Discriminator(NUM_CLASSES + REPRESENT_DIM, HIDDEN, 1)

BATCH_SIZE = 128
G_Optim = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
D_Optim = torch.optim.Adam(D.parameters(), lr=1e-4, betas=(0.5, 0.999))

MEANS = torch.tensor([-2.0, 2.0])
STD = 0.4


def sample_real_conditional(n, num_classes):
    c = torch.randint(0, num_classes, (n,))
    x = MEANS[c].unsqueeze(-1) + torch.randn(n, 1) * STD
    return x, c


criterion = nn.BCEWithLogitsLoss()

for step in range(1, 8001):
    reals_t, cs_t = sample_real_conditional(BATCH_SIZE, NUM_CLASSES)
    noise = torch.randn(BATCH_SIZE, Z_DIM)

    D.train()
    D_Optim.zero_grad(set_to_none=True)
    real_logits = D(reals_t, cs_t, NUM_CLASSES)
    with torch.no_grad():
        fakes = G(noise, cs_t, NUM_CLASSES)
    fake_logits = D(fakes.detach(), cs_t, NUM_CLASSES)
    loss_D = criterion(real_logits, torch.ones_like(real_logits)) + criterion(
        fake_logits, torch.zeros_like(fake_logits)
    )
    loss_D.backward()
    D_Optim.step()

    for _ in range(2):
        noise = torch.randn(BATCH_SIZE, Z_DIM)
        G.train()
        G_Optim.zero_grad(set_to_none=True)
        fake_logits = D(G(noise, cs_t, NUM_CLASSES), cs_t, NUM_CLASSES)
        loss_G = criterion(fake_logits, torch.ones_like(fake_logits))
        loss_G.backward()
        G_Optim.step()

    if step % 500 == 0:
        with torch.no_grad():
            p_real = torch.sigmoid(D(reals_t, cs_t, NUM_CLASSES)).mean().item()
            p_fake = torch.sigmoid(D(fakes, cs_t, NUM_CLASSES)).mean().item()
            m0 = G(torch.randn(500, Z_DIM), torch.zeros(500, dtype=torch.long), NUM_CLASSES).mean().item()
            m1 = G(torch.randn(500, Z_DIM), torch.ones(500, dtype=torch.long), NUM_CLASSES).mean().item()
        print(
            f"step {step}: loss_D {loss_D.item():.3f}, loss_G {loss_G.item():.3f} | "
            f"D(real)={p_real:.2f} D(fake)={p_fake:.2f} | "
            f"class 0 mean {m0:+.2f}  class 1 mean {m1:+.2f}"
        )

step 500: loss_D 1.388, loss_G 0.698 | D(real)=0.50 D(fake)=0.50 | class 0 mean -1.88  class 1 mean +2.02
step 1000: loss_D 1.387, loss_G 0.697 | D(real)=0.50 D(fake)=0.50 | class 0 mean -2.02  class 1 mean +2.07
step 1500: loss_D 1.384, loss_G 0.678 | D(real)=0.51 D(fake)=0.51 | class 0 mean -1.47  class 1 mean +1.87
step 2000: loss_D 1.383, loss_G 0.679 | D(real)=0.51 D(fake)=0.51 | class 0 mean -1.47  class 1 mean +1.89
step 2500: loss_D 1.388, loss_G 0.691 | D(real)=0.50 D(fake)=0.50 | class 0 mean -1.39  class 1 mean +2.43
step 3000: loss_D 1.390, loss_G 0.684 | D(real)=0.50 D(fake)=0.50 | class 0 mean -2.13  class 1 mean +1.88
step 3500: loss_D 1.382, loss_G 0.703 | D(real)=0.50 D(fake)=0.50 | class 0 mean -2.63  class 1 mean +1.38
step 4000: loss_D 1.389, loss_G 0.701 | D(real)=0.50 D(fake)=0.50 | class 0 mean -1.90  class 1 mean +2.36
step 4500: loss_D 1.385, loss_G 0.690 | D(real)=0.50 D(fake)=0.50 | class 0 mean -1.75  class 1 mean +2.22
step 5000: loss_D 1.387, loss_G 0.691 

### 实验：不用 one-hot，喂 `[0.5, 0.5]` 会怎样？

训练时 G 只见过 **硬标签** `[1,0]` 和 `[0,1]`，从未见过软标签。推理时把条件换成 `[0.5, 0.5]`（「50% class 0 + 50% class 1」），看 G 会生成什么——这相当于在两种 one-hot 输入之间做**线性插值**，输出通常会落在两个 mode 的中间，而不是随机混出双峰。

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

import matplotlib.pyplot as plt
import numpy as np


def G_with_label(z, label):
    """直接用 label 向量（one-hot 或 soft）拼 z，绕过 one_hot(c)。"""
    return G.net(torch.cat([z, label], dim=-1))


n = 3000
z = torch.randn(n, Z_DIM)

labels = {
    "one_hot(0) [1,0]": torch.tensor([[1.0, 0.0]]).expand(n, -1),
    "one_hot(1) [0,1]": torch.tensor([[0.0, 1.0]]).expand(n, -1),
    "soft [0.5, 0.5]": torch.tensor([[0.5, 0.5]]).expand(n, -1),
}

G.eval()
with torch.no_grad():
    samples = {name: G_with_label(z, label).squeeze().numpy() for name, label in labels.items()}

with SectionPrinter("one-hot vs soft label"):
    for name, xs in samples.items():
        print(f"{name:22s}  mean {xs.mean():+.2f}  std {xs.std():.2f}")

    fig, ax = plt.subplots(figsize=(8, 4))
    bins = np.linspace(-4, 4, 40)
    for name, xs in samples.items():
        ax.hist(xs, bins=bins, alpha=0.45, density=True, label=name)
    ax.axvline(-2, color="gray", ls="--", lw=1)
    ax.axvline(0, color="gray", ls=":", lw=1)
    ax.axvline(2, color="gray", ls="--", lw=1)
    ax.set_xlabel("x")
    ax.set_ylabel("density")
    ax.set_title("G(z, label): hard vs soft condition")
    ax.legend()
    plt.tight_layout()
    plt.show()